In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CONCATENATION DES PANELS SOLDE 2015-2020 ET 2021-2025
"""
import boto3
import pandas as pd
from io import BytesIO

# =============================================================================
# CONNEXION S3
# =============================================================================
print("="*80)
print("🔗 CONCATENATION DES PANELS SOLDE")
print("="*80 + "\n")
print("⚙️  Connexion à MinIO/S3...")

s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)
bucket_name = "admindataanstat"
print("✅ Connexion établie\n")

# =============================================================================
# CHARGEMENT DES DEUX BASES
# =============================================================================
print("📂 Chargement des fichiers Parquet...\n")

# Fichier 1 : 2015-2020
file1_key = "Solde/Panel_solde_1_2_2015_2020.parquet"
print(f"🔄 Lecture: {file1_key}")
try:
    obj1 = s3_client.get_object(Bucket=bucket_name, Key=file1_key)
    df1 = pd.read_parquet(BytesIO(obj1["Body"].read()))
    print(f"✅ Panel 2015-2020: {len(df1):,} lignes, {df1.shape[1]} colonnes")
    print(f"   Colonnes: {list(df1.columns)[:5]}...")
    
    # Identifier la période (en gérant les valeurs nulles)
    if 'PERIODE' in df1.columns:
        periodes1 = df1['PERIODE'].dropna().unique()
        if len(periodes1) > 0:
            print(f"   Périodes: {len(periodes1)} mois ({min(periodes1)} → {max(periodes1)})")
        else:
            print(f"   ⚠️  Colonne PERIODE vide ou que des valeurs nulles")
        
        # Vérifier les valeurs nulles
        nb_null = df1['PERIODE'].isna().sum()
        if nb_null > 0:
            print(f"   ⚠️  {nb_null:,} valeurs nulles dans PERIODE ({nb_null/len(df1)*100:.2f}%)")
    
except Exception as e:
    print(f"❌ Erreur lecture fichier 1: {e}")
    import traceback
    traceback.print_exc()
    exit(1)

# Fichier 2 : 2021-2025
file2_key = "Solde/Panel_solde_1_2_2021_2025.parquet"  # ✅ Ajout de .parquet
print(f"\n🔄 Lecture: {file2_key}")
try:
    obj2 = s3_client.get_object(Bucket=bucket_name, Key=file2_key)
    df2 = pd.read_parquet(BytesIO(obj2["Body"].read()))
    print(f"✅ Panel 2021-2025: {len(df2):,} lignes, {df2.shape[1]} colonnes")
    print(f"   Colonnes: {list(df2.columns)[:5]}...")
    
    # Identifier la période (en gérant les valeurs nulles)
    if 'PERIODE' in df2.columns:
        periodes2 = df2['PERIODE'].dropna().unique()
        if len(periodes2) > 0:
            print(f"   Périodes: {len(periodes2)} mois ({min(periodes2)} → {max(periodes2)})")
        else:
            print(f"   ⚠️  Colonne PERIODE vide ou que des valeurs nulles")
        
        # Vérifier les valeurs nulles
        nb_null = df2['PERIODE'].isna().sum()
        if nb_null > 0:
            print(f"   ⚠️  {nb_null:,} valeurs nulles dans PERIODE ({nb_null/len(df2)*100:.2f}%)")
    
except Exception as e:
    print(f"❌ Erreur lecture fichier 2: {e}")
    import traceback
    traceback.print_exc()
    exit(1)

# =============================================================================
# VERIFICATION DE LA COMPATIBILITE DES SCHEMAS
# =============================================================================
print("\n" + "="*80)
print("🔍 VERIFICATION DES SCHEMAS")
print("="*80 + "\n")

# Comparer les colonnes
cols1 = set(df1.columns)
cols2 = set(df2.columns)

communes = cols1 & cols2
uniquement_df1 = cols1 - cols2
uniquement_df2 = cols2 - cols1

print(f"📊 Colonnes communes: {len(communes)}")
print(f"📊 Uniquement 2015-2020: {len(uniquement_df1)}")
if uniquement_df1:
    print(f"    {list(uniquement_df1)[:10]}...")
print(f"📊 Uniquement 2021-2025: {len(uniquement_df2)}")
if uniquement_df2:
    print(f"    {list(uniquement_df2)[:10]}...")

# =============================================================================
# CONCATENATION
# =============================================================================
print("\n" + "="*80)
print("🔗 CONCATENATION EN COURS...")
print("="*80 + "\n")

try:
    df_concat = pd.concat([df1, df2], ignore_index=True)
    print(f"✅ Concaténation réussie!")
    print(f"   Total: {len(df_concat):,} lignes, {df_concat.shape[1]} colonnes")
    
    if 'PERIODE' in df_concat.columns:
        periodes_finales = df_concat['PERIODE'].dropna().unique()
        if len(periodes_finales) > 0:
            print(f"   Périodes: {len(periodes_finales)} mois ({min(periodes_finales)} → {max(periodes_finales)})")
    
except Exception as e:
    print(f"❌ Erreur lors de la concaténation: {e}")
    import traceback
    traceback.print_exc()
    exit(1)

# =============================================================================
# SAUVEGARDE
# =============================================================================
print("\n" + "="*80)
print("💾 SAUVEGARDE DU PANEL COMPLET")
print("="*80 + "\n")

output_key = "Solde/Panel_solde_complet_2015_2025.parquet"
print(f"📤 Écriture vers: {output_key}")

try:
    buffer = BytesIO()
    df_concat.to_parquet(buffer, index=False, engine='pyarrow', compression='snappy')
    buffer.seek(0)
    
    s3_client.put_object(
        Bucket=bucket_name,
        Key=output_key,
        Body=buffer.getvalue()
    )
    
    print(f"✅ Fichier sauvegardé avec succès!")
    print(f"   Taille finale: {len(df_concat):,} lignes")
    
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde: {e}")
    import traceback
    traceback.print_exc()
    exit(1)

print("\n" + "="*80)
print("✅ TRAITEMENT TERMINÉ")
print("="*80)